**Auto Loader**
- Recommended alternative to legacy COPY INTO
- SQL version of AUTO LOADER works only on warehouse compute
- Python version of AUTO LOADER works on All Purpose cluster but needs to have dedicated access
- Python version requires to manually set file location for checkpoint data
- each autloader should have its own checkpoint folder

In [0]:
DROP TABLE IF EXISTS data_fusion.bronze_delta_tables.orders_incremental_load_autoloader_SQL;
DROP TABLE IF EXISTS data_fusion.bronze_delta_tables.orders_incremental_load_autoloader_python;

#### SQL Version

In [0]:
-- SQL version of AUTO LOADER
CREATE OR REFRESH STREAMING TABLE data_fusion.bronze_delta_tables.orders_incremental_load_autoloader_SQL
AS 
SELECT
     * 
FROM STREAM read_files(
    '/Volumes/data_fusion/source_data/incremental_ingestion/Orders_incrementaly_loaded_CSV/',
    FORMAT => 'CSV'
)


#### Python Version

In [0]:
%python
# creating directory for checkpoint Data
dbutils.fs.mkdirs('/Volumes/data_fusion/source_data/incremental_ingestion/Checkpoint_files/Orders_checkpoint')

In [0]:
%python
checkpoint_location = '/Volumes/data_fusion/source_data/incremental_ingestion/Checkpoint_files/Orders_checkpoint/'

df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option("cloudFiles.schemaLocation",checkpoint_location)
        .load('/Volumes/data_fusion/source_data/incremental_ingestion/Orders_incrementaly_loaded_CSV/')
        )

df.writeStream.option("checkpointLocation", checkpoint_location) \
              .trigger(once=True) \
              .toTable('data_fusion.bronze_delta_tables.orders_incremental_load_autoloader_Python')